<a href="https://colab.research.google.com/github/rajeshsharma38/neural-network-learning/blob/main/MINST_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd

In [80]:
import torch.nn.functional as F

In [90]:
df = pd.read_csv('mnist_train_small.csv', header=None)
df

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [115]:
df.shape
Ytr = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
Xtr = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32)
Xtr.shape, Ytr.shape, Xtr.dtype, Ytr.dtype

(torch.Size([1000, 784]), torch.Size([1000]), torch.float32, torch.int64)

In [116]:
w1 = torch.randn(784, 100) / 784**0.5
b1 = torch.randn(100)
w2 = torch.randn(100, 10) / 100**0.5
b2 = torch.randn(10)

parameters = [w1, b1, w2, b2]

for p in parameters:
  p.requires_grad = True


In [117]:
X_mean = Xtr.mean(0, keepdim=True)
X_std = Xtr.std(0, keepdim=True)
eps = 0.01

Xtr = (Xtr - X_mean) / (X_std + eps)

In [118]:
#forward pass
for i in range(100): # Reduced iterations for faster debugging
  hpreact = Xtr @ w1 + b1

  # Detach, convert to numpy, flatten, and then remove NaNs before plotting
  # import numpy as np # Import numpy if not already imported
  # hpreact_flat = hpreact.detach().numpy().flatten()
  # hpreact_flat = hpreact_flat[~np.isnan(hpreact_flat)] # Filter out NaN values

  # if hpreact_flat.size > 0: # Only plot if there's data after removing NaNs
  #   plt.hist(hpreact_flat, bins=200) # Plot a histogram of flattened hpreact values
  #   plt.title('Distribution of hpreact values (NaNs removed)') # Add a title
  #   plt.xlabel('Value') # Add x-axis label
  #   plt.ylabel('Frequency') # Add y-axis label
  #   plt.show() # Ensure the plot is displayed
  # else:
  #   print("No valid data to plot after removing NaNs from hpreact.")

  # break # The break statement is currently preventing the training loop from running

  h = torch.tanh(hpreact)

  logits = h @ w2 + b2
  loss = F.cross_entropy(logits, Ytr)

  #print(loss.item())

  for p in parameters:
    p.grad = None

  loss.backward()

  for p in parameters:
    p.data += -0.1 * p.grad

  if i%10 == 0: # Print loss more frequently with fewer iterations
    print(loss.item())

2.840010166168213
0.9935153126716614
0.6562259197235107
0.4998793601989746
0.40526166558265686
0.34035632014274597
0.292601615190506
0.2558533251285553
0.2265971153974533
0.20267270505428314


Now that you've trained the model, let's use it to predict digits on a small portion of the test set (`mnist_test.csv`). We'll take the first 10 examples to quickly see how it performs.

In [95]:
# Load the test dataset (mnist_test.csv)
df_test = pd.read_csv('mnist_test.csv', header=None)

# Take only the first 10 examples for prediction
X_test = torch.tensor(df_test.iloc[:10, 1:].values, dtype=torch.float32)
Y_test = torch.tensor(df_test.iloc[:10, 0].values, dtype=torch.long)

print(f"Test data shape (features): {X_test.shape}")
print(f"Test data shape (labels): {Y_test.shape}")

Test data shape (features): torch.Size([10, 784])
Test data shape (labels): torch.Size([10])


In [97]:
X_test = (X_test - X_mean) / (X_std + eps)

In [98]:
# Perform forward pass on the test data using the trained weights (w1, b1, w2, b2)
hpreact_test = X_test @ w1 + b1
h_test = torch.tanh(hpreact_test)
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

print("--- Predictions vs Actual Labels (First 10 Examples) ---")
for i in range(len(predictions)):
    print(f"Example {i+1}: Predicted = {predictions[i].item()}, Actual = {Y_test[i].item()}")

--- Predictions vs Actual Labels (First 10 Examples) ---
Example 1: Predicted = 7, Actual = 7
Example 2: Predicted = 5, Actual = 2
Example 3: Predicted = 1, Actual = 1
Example 4: Predicted = 0, Actual = 0
Example 5: Predicted = 7, Actual = 4
Example 6: Predicted = 1, Actual = 1
Example 7: Predicted = 4, Actual = 4
Example 8: Predicted = 5, Actual = 9
Example 9: Predicted = 8, Actual = 5
Example 10: Predicted = 9, Actual = 9
